In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [49]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [6]:
loader = PyPDFLoader("Submission.pdf")
doc = loader.load()
print(len(doc))

7


In [9]:
text_spliter= RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_text= text_spliter.split_documents(doc)
len(splitted_text)


12

In [14]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")


In [15]:
vector_store = Chroma.from_documents(
    embedding=embeddings,
    documents=splitted_text,
    persist_directory="./vectorEmbedding.db"
)

In [43]:
query = input("LLM")
data = vector_store.similarity_search(query=query)
print(len(data))
#print(data[0].page_content)

4


In [48]:
context = ""
for doc in data:
    context += doc.page_content + "\n"


In [45]:
llm = ChatOpenAI(model="gpt-4o")

In [46]:
res = llm.invoke(f"based on the context priovided can you find answers for my question, context : {context} and question is : {query}")

In [47]:
print(res.content)

The context you provided is about a project called "Dark Phoenix" which is an AI podcast clipper. It appears to be an end-to-end AI pipeline designed to transform long-form podcast videos into short-form, watermarked, and captioned clips suitable for social media distribution. Here are some details related to AI and LLM (Large Language Models) from the context:

1. **AI Podcast Clipping**: The project automates the process of creating short video clips from longer podcast videos using AI technology. This involves tasks like moment selection and speaker detection which are typically AI-driven processes.

2. **LLM for Moment Selection**: The project uses a combination of large language models (LLM) for selecting moments to clip. Primary model is "Gemini 2.5-flash" with a fallback on "NVIDIA NIM Llama3.3-70b". This indicates the use of sophisticated language models to understand the content of the podcast and identify key moments to clip.

3. **Transcript Sampling**: The project handles l

In [50]:
def get_context(query : str):
    data = vector_store.similarity_search(query=query)
    context = ""
    for doc in data:
        context += doc.page_content + "\n"

        return {
            "context" : context,
            "question": query
        }

In [52]:
prompt = PromptTemplate.from_template(
    """
        you are a ai assigstant which help to get answers based on the context provided and question.
        and if you dont know the answers then can you say "I dont know the answer or i am not able to find context from pdf."
        args :
               context : {context}
                question : {question}
        """
)

In [53]:
rag_chain = get_context | prompt | llm

In [72]:
res= rag_chain.invoke("summarize me content from context")

In [73]:
print(res.content)

The context provided describes a project named "Dark Phoenix — AI Podcast Clipper," which is an AI-based system designed to transform long-form podcast videos into short, watermarked, captioned clips suitable for social media distribution. This project involves various files and updates, including a README.md file with recent additions, and other supporting documents that detail improvements in clip quality and other functionalities. The project is hosted on a platform with a live app link provided, and offers features like uploading a podcast video or pasting a YouTube URL for automated processing. The system is likely managed on a repository with functionalities for issues tracking, pull requests, and more. Additionally, there's a reviewer login mentioned for the platform.
